# 序列逆置
使用sequence to sequence 模型将一个字符串序列逆置。
例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个sequence to sequence 模型示意图 )
![seq2seq](./seq2seq.png)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['SRDGDIMBJO', 'HMRSLWVVGD'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[19, 18,  4,  7,  4,  9, 13,  2, 10, 15],
       [ 8, 13, 18, 19, 12, 23, 22, 22,  7,  4]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 15, 10,  2, 13,  9,  4,  7,  4, 18],
       [ 0,  4,  7, 22, 22, 23, 12, 19, 18, 13]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[15, 10,  2, 13,  9,  4,  7,  4, 18, 19],
       [ 4,  7, 22, 22, 23, 12, 19, 18, 13,  8]])>)


# 建立sequence to sequence 模型

In [28]:
class mySeq2SeqModel(tf.keras.Model):
    def __init__(self, vocab_size=30, emb_dim=64, hidden_size=128):
        super().__init__()

        self.embed_layer = layers.Embedding(vocab_size, emb_dim)
        self.encoder = layers.SimpleRNN(
            hidden_size,
            return_sequences=True,
            return_state=True
        )

        self.decoder_cell = layers.SimpleRNNCell(hidden_size)

        self.dense = layers.Dense(vocab_size)

    # 编码
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids)

        enc_out, enc_state = self.encoder(enc_emb)

        # 强制保证 shape = [batch, hidden]
        enc_state = tf.reshape(enc_state, [tf.shape(enc_ids)[0], -1])

        return enc_out, enc_state

    # 训练 forward
    @tf.function
    def call(self, enc_ids, dec_ids):
        enc_out, enc_state = self.encode(enc_ids)

        dec_emb = self.embed_layer(dec_ids)

        dec_len = tf.shape(dec_ids)[1]
        state = enc_state

        outputs = tf.TensorArray(tf.float32, size=dec_len)

        for t in tf.range(dec_len):
            x_t = dec_emb[:, t, :]

            out, state = self.decoder_cell(x_t, state)

            logit = self.dense(out)

            outputs = outputs.write(t, logit)

        logits = outputs.stack()             # [T,B,V]
        logits = tf.transpose(logits, [1,0,2])  # [B,T,V]

        return logits

    # 推理一步
    def get_next_token(self, x, state):
        x = self.embed_layer(x)

        state = tf.reshape(state, [tf.shape(x)[0], -1])

        out, state = self.decoder_cell(x, state)

        logits = self.dense(out)

        return logits, state

# Loss函数以及训练逻辑

In [29]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(3000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [30]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.4114075
step 500 : loss 1.6812794
step 1000 : loss 1.0367001
step 1500 : loss 0.7652009
step 2000 : loss 0.52941716
step 2500 : loss 0.46351376


<tf.Tensor: shape=(), dtype=float32, numpy=0.37133765>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [31]:
def sequence_reversal():
    def decode(init_state, steps):
        b_sz = tf.shape(init_state)[0]

        cur_token = tf.zeros([b_sz], dtype=tf.int32)
        state = init_state

        outputs = []

        for _ in range(steps):
            logits, state = model.get_next_token(cur_token, state)

            cur_token = tf.argmax(logits, axis=-1, output_type=tf.int32)

            outputs.append(cur_token)

        return tf.stack(outputs, axis=1)

    # 测试
    batched_examples, enc_x, dec_x, y = get_batch(2, 10)

    enc_out, init_state = model.encode(enc_x)

    print("init_state shape:", init_state.shape)  # 应该是 (2,128)

    pred = decode(init_state, 10)

    print("input:", enc_x.numpy())
    print("pred :", pred.numpy())
    print("true :", y.numpy())

    return enc_x.numpy(), pred.numpy(), y.numpy()

def is_reverse(inp, pred, true):
    return (pred == true).all()

print([is_reverse(*item) for item in zip(*sequence_reversal())])
print(list(zip(*sequence_reversal())))

init_state shape: (2, 128)
input: [[25  6  2 14  8 19 25  9 22 15]
 [17 15 13 22  9 11  4  2 26  4]]
pred : [[15  9  9 25  4  6 22  6 19 14]
 [ 4 26  2  4 11  9 22 13 15 17]]
true : [[15 22  9 25 19  8 14  2  6 25]
 [ 4 26  2  4 11  9 22 13 15 17]]
[False, True]
init_state shape: (2, 128)
input: [[ 6 12 26 14 16 22  7 22 21 10]
 [ 2  4 11 13 22  4 21 23  7 15]]
pred : [[10 21 22  7 22 16 14 26 12  6]
 [15  7 23 21  4 22 13 11  4  2]]
true : [[10 21 22  7 22 16 14 26 12  6]
 [15  7 23 21  4 22 13 11  4  2]]
[(array([ 6, 12, 26, 14, 16, 22,  7, 22, 21, 10]), array([10, 21, 22,  7, 22, 16, 14, 26, 12,  6]), array([10, 21, 22,  7, 22, 16, 14, 26, 12,  6])), (array([ 2,  4, 11, 13, 22,  4, 21, 23,  7, 15]), array([15,  7, 23, 21,  4, 22, 13, 11,  4,  2]), array([15,  7, 23, 21,  4, 22, 13, 11,  4,  2]))]
